In [1]:
!git clone https://github.com/dave123981/commerce-recommendation-engine.git
%cd commerce-recommendation-engine
!pip install -r requirements.txt -q

fatal: destination path 'commerce-recommendation-engine' already exists and is not an empty directory.
/content/commerce-recommendation-engine


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Path to dataset files: /kaggle/input/brazilian-ecommerce


In [4]:
import pandas as pd

# example for Olist — adjust column names to your dataset
orders = pd.read_csv("/kaggle/input/brazilian-ecommerce/olist_orders_dataset.csv", parse_dates=["order_purchase_timestamp"])
items = pd.read_csv("/kaggle/input/brazilian-ecommerce/olist_order_items_dataset.csv")

interactions = (
    items.merge(orders[["order_id", "customer_id", "order_purchase_timestamp"]], on="order_id")
    .rename(columns={
        "customer_id": "user_id",
        "product_id": "item_id",
        "order_purchase_timestamp": "timestamp",
    })[["user_id", "item_id", "order_id", "timestamp"]]
)
interactions["quantity"] = 1
interactions.to_csv("interactions.csv", index=False)

In [5]:
from src.metrics import time_based_split

interactions = pd.read_csv("interactions.csv", parse_dates=["timestamp"])
train, test = time_based_split(interactions, test_frac=0.2)
train.to_csv("train.csv", index=False)
test.to_csv("test.csv", index=False)

In [ ]:
from src.v1_popularity import PopularityRecommender
from src.metrics import evaluate_model

model = PopularityRecommender(half_life_days=30).fit(train)
metrics = evaluate_model(model, test, k=10)
print(metrics)